In [5]:
import warnings
warnings.filterwarnings("ignore")

import networkx as nx

try:
    G = nx.read_gml("polblogs.gml", label="id")
    print("Loaded as simple DiGraph.")
except nx.NetworkXError as e:
    print(f"Duplicate edges found ({e}). Loading as MultiDiGraph and collapsing...")
    # NetworkX's read_gml can't handle duplicates into a simple graph,
    # so parse it manually via the multi-graph reader route:
    import gzip
    with open("polblogs.gml", "r", encoding="utf-8", errors="replace") as f:
        raw = f.read()
    # Force multi-graph by reading raw GML through igraph, then converting
    import igraph as ig
    g_ig = ig.Graph.Read_GML("polblogs.gml")
    G = nx.DiGraph()
    # Add nodes with attributes
    for v in g_ig.vs:
        node_id = int(v["id"]) if "id" in v.attributes() else v.index
        attrs = {k: v[k] for k in v.attributes() if v[k] is not None}
        attrs["label"] = v["label"] if "label" in v.attributes() else str(node_id)
        G.add_node(node_id, **attrs)
    # Add edges (igraph collapses duplicates automatically into a simple graph)
    for e in g_ig.es:
        s = int(g_ig.vs[e.source]["id"])
        t = int(g_ig.vs[e.target]["id"])
        G.add_edge(s, t)
    print("Loaded via igraph and collapsed duplicates.")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Directed:", G.is_directed())

Duplicate edges found (edge #12757 (1047->1179) is duplicated). Loading as MultiDiGraph and collapsing...
Loaded via igraph and collapsed duplicates.
Nodes: 1490
Edges: 19025
Directed: True
